|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 0:</h2>|<h1>From a Program to a Model<h1>|
|<h2>Section:</h2>|<h1>Incidents<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: break it on purpose<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT/'course/Part0_FromAProgramToAModel/4_incidents'))

import math, time
import torch
import lab
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

In the incident file you went from a symptom to a cause. Here you go the other
way. You put one fault into a working model, a working loop or a working
kernel, and you watch what it does.

The routine for each exercise is the same:

1. Read the fault.
2. **Write your prediction in the cell.** Answer the four questions.
3. Run the cell.
4. Write down where your prediction was wrong. This line is the one that
   teaches you.

The four questions:

- **Crash?** Does it raise an error, or does it run?
- **When?** When does the output first differ from the reference: which
  token, which prompt, which size?
- **What?** What does the wrong result look like?
- **Which guard?** Which check would catch it?

The reference is a correct greedy loop in `lab.py`, on the attention of HF.
`lab.use_attention(model, fn)` puts your own attention function into every
layer of the model, and `lab.use_attention(model, None)` takes it out.

This notebook needs a GPU with about 8 GB free.

In [ ]:
### run this cell

MODEL = 'Qwen/Qwen3-1.7B'
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).cuda().eval()
device = 'cuda'

PROMPT = 'The three largest cities in Japan are'
TOKENS = 30

def report(name, generated, expected):
  lab.report(tokenizer, name, generated, expected)

expected = lab.greedy(model, tokenizer, PROMPT, TOKENS)
report('reference', expected, expected)

# Exercise 1: the tokenizer of another model

Encode the prompt with the tokenizer of Llama-2, feed the ids to Qwen3, and
decode the answer with the same wrong tokenizer. Then do it again with the
tokenizer of Qwen2.5, an older model of the same family.

This is Ticket 1 of the incident file. Predict both results. Does the second
tokenizer break anything?

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
llama = AutoTokenizer.from_pretrained('hf-internal-testing/llama-tokenizer')   # the Llama-2 tokenizer
older = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B')
print('entries:  Llama-2', len(llama), '  Qwen2.5', len(older), '  Qwen3', len(tokenizer),
      '  the model', model.config.vocab_size)

for name, wrong in [('Llama-2', llama), ('Qwen2.5', older)]:
  ids = wrong(PROMPT, return_tensors='pt').input_ids.to(device)       # THE FAULT
  print(f'\n{name} ids: {ids[0].tolist()}')
  print('  what Qwen3 reads:', repr(tokenizer.decode(ids[0])))
  generated = lab.greedy_ids(model, ids, TOKENS)
  print('  largest id written:', max(generated))
  try:
    print('  decoded with the wrong tokenizer:', repr(wrong.decode(generated)))
  except Exception as error:
    print('  decode raised', type(error).__name__, error)

# Exercise 2: exp() in float16

Load a second copy of the model in float16. First run it with the attention
of HF. Then put in an attention that calls `torch.exp` on the raw scores,
with no subtraction of the maximum. Run a short prompt and a long one, and
record the largest score that each layer sees.

This is Ticket 2. Predict which prompt fails, and what the text looks like.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
model16 = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).cuda().eval()
LONG = ('Summarize the following report. ' + 'The committee met on Tuesday to review the budget, '
        'the hiring plan and the new office lease, and it approved all three with minor changes. ' * 12)
largest = []

def naive_exp_attention(q, k, v):
  scores = q @ k.transpose(-1, -2) / math.sqrt(q.shape[-1])
  scores = scores + lab.causal_bias(scores)
  largest.append(scores.max().item())
  weights = torch.exp(scores)                                        # THE FAULT: no max subtracted
  weights = weights / weights.sum(-1, keepdim=True)
  return weights @ v

for name, prompt in [('short', PROMPT), ('long', LONG)]:
  ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
  lab.use_attention(model16, None)
  hf = lab.greedy_ids(model16, ids, 15)
  lab.use_attention(model16, naive_exp_attention)
  largest.clear()
  with torch.inference_mode():
    logits = model16(ids).logits
  layer_max = largest[:model.config.num_hidden_layers]
  naive = lab.greedy_ids(model16, ids, 15)
  print(f'{name}: {ids.shape[1]} tokens, largest score {max(layer_max):.1f} '
        f'(layer {layer_max.index(max(layer_max))}), finite logits: {torch.isfinite(logits).all().item()}')
  print('   HF float16:   ', repr(tokenizer.decode(hf)))
  print('   naive float16:', repr(tokenizer.decode(naive)))
print('exp() overflows float16 above', round(math.log(torch.finfo(torch.float16).max), 2))
lab.use_attention(model16, None)
del model16; torch.cuda.empty_cache()

# Exercise 3: the attention with no causal mask

The function forgets the causal mask. First run the unit test of Ticket 3:
one query against 50 keys. Then run the same test with 20 queries. Then put
the function into the model.

Predict: at which token does the model go wrong? The loop has a KV cache, so
every decode step has exactly one query.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
def no_mask_attention(q, k, v):
  scores = q @ k.transpose(-1, -2) / math.sqrt(q.shape[-1])
  return torch.softmax(scores.float(), -1).to(q.dtype) @ v            # THE FAULT: no mask

torch.manual_seed(0)
for queries in (1, 20):
  q = torch.randn(1, 16, queries, 128, device=device)
  k = torch.randn(1, 16, 50, 128, device=device)
  v = torch.randn(1, 16, 50, 128, device=device)
  want = lab.correct_attention(q, k, v)
  print(f'unit test with {queries:2d} queries: largest difference {(no_mask_attention(q, k, v) - want).abs().max().item():.4f}')

lab.use_attention(model, no_mask_attention)
report('no mask', lab.greedy(model, tokenizer, PROMPT, TOKENS), expected)
lab.use_attention(model, None)

# Exercise 4: the kernel at a fraction of a percent of peak

Time the RMSNorm of the model on 32,768 tokens x 2,048 values. Report it in
TFLOP/s, as the team of Ticket 4 did, and in GB/s. Compare it with a plain
copy of the same tensor, with the same RMSNorm after `torch.compile`, and
with the numbers of `./vc info`.

The tensor is 134 MB, larger than the L2 cache of the card (`./vc info`). A
tensor that fits in L2 does not measure the memory.

Predict the fractions: of the compute peak, and of the copy bandwidth.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
PEAK_TFLOPS, PEAK_GBS = 49, 294        # the sustained numbers of ./vc info on your card

x = torch.randn(32768, 2048, dtype=torch.bfloat16, device=device)
norm = model.model.norm
compiled = torch.compile(norm)

@torch.inference_mode()
def ms_per_call(fn, iters=50):
  for _ in range(5):
    fn()
  torch.cuda.synchronize()
  start = time.perf_counter()
  for _ in range(iters):
    fn()
  torch.cuda.synchronize()
  return (time.perf_counter() - start) / iters * 1000

bytes_moved = 2 * x.numel() * x.element_size()           # read x, write y
flop = 4 * x.numel()
for name, fn in [('RMSNorm, HF eager', lambda: norm(x)), ('RMSNorm, compiled', lambda: compiled(x)),
                 ('clone            ', lambda: x.clone())]:
  ms = ms_per_call(fn)
  print(f'{name} {ms:.3f} ms   {flop / ms / 1e9:6.3f} TFLOP/s = {flop / ms / 1e9 / PEAK_TFLOPS:.2%} of peak   '
        f'{bytes_moved / ms / 1e6:5.0f} GB/s = {bytes_moved / ms / 1e6 / PEAK_GBS:.0%} of ./vc info')

# Exercise 5: load with no dtype

Load the model on the CPU two times: once with no dtype, as the code of
Ticket 5 does in an old version of transformers, and once in bfloat16. Count
the bytes of the weights, and the bytes of one MLP matrix.

Predict the two totals before you run. The model has 1.72 B parameters.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
def model_bytes(m):
  return sum(p.numel() * p.element_size() for p in m.parameters())

for dtype in (torch.float32, torch.bfloat16):                       # float32: THE FAULT of the old default
  cpu_model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype)
  matrix = cpu_model.model.layers[0].mlp.up_proj.weight
  print(f'{str(dtype):15s} {model_bytes(cpu_model) / 1e9:5.2f} GB, one MLP matrix {tuple(matrix.shape)} = '
        f'{matrix.numel() * matrix.element_size() / 2**20:.0f} MiB')
  del cpu_model

# Exercise 6: generate() with no sampling arguments

Call `model.generate` three times with only `max_new_tokens`, as the harness
of Ticket 6 does. Then three times with `torch.manual_seed(0)` before each
call. Then with `do_sample=False`.

Predict which of the three groups agree with each other, and which one
agrees with the greedy reference.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
ids = tokenizer(PROMPT, return_tensors='pt').to(device)
print('generation_config:', {k: getattr(model.generation_config, k)
                             for k in ('do_sample', 'temperature', 'top_k', 'top_p')})

def run(**kwargs):
  out = model.generate(**ids, max_new_tokens=TOKENS, **kwargs)        # THE FAULT: no do_sample
  return out[0, ids.input_ids.shape[1]:].tolist()

plain = [run() for _ in range(3)]
seeded = []
for _ in range(3):
  torch.manual_seed(0)
  seeded.append(run())
greedy = run(do_sample=False)
for name, group in [('no arguments', plain), ('seed 0 each time', seeded)]:
  print(f'{name}: {len({tuple(g) for g in group})} different answers of 3; first differences with the reference:',
        [lab.first_difference(g, expected) for g in group])
report('do_sample=False', greedy, expected)

# Exercise 7: generate() with no length

Call `model.generate` with no length at all, for prompts of several lengths.
Count the prompt tokens and the new tokens.

This is Ticket 7. Predict the new tokens for each prompt.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
import warnings
for prompt in ['Hello', 'The three largest cities in Japan are',
               'Write a long story about a robot who learns to paint, and about its first exhibition']:
  ids = tokenizer(prompt, return_tensors='pt').to(device)
  with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    out = model.generate(**ids, do_sample=False)                      # THE FAULT: no max_new_tokens
  prompt_len = ids.input_ids.shape[1]
  new = out.shape[1] - prompt_len
  print(f'prompt {prompt_len:2d} + new {new:2d} = {prompt_len + new}')
print('warnings:', {str(w.message)[:90] for w in caught})

# Exercise 8: the chat model with no chat template

Send a question as raw text, then the same question in the chat template.

This is Ticket 8. Predict the first words of each answer.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
question = 'What is the capital of France?'
raw = tokenizer(question, return_tensors='pt').input_ids.to(device)             # THE FAULT
chat = tokenizer.apply_chat_template([{'role': 'user', 'content': question}],
                                     add_generation_prompt=True, enable_thinking=False,
                                     return_tensors='pt', return_dict=True).input_ids.to(device)
stop = set(model.generation_config.eos_token_id)
for name, ids in [('raw text', raw), ('chat template', chat)]:
  count = (ids == 151644).sum().item()
  answer = lab.greedy_ids(model, ids, 40, stop)
  print(f'{name}: {ids.shape[1]} prompt tokens, <|im_start|> appears {count} times')
  print('   ', repr(tokenizer.decode(answer)))

# Exercise 9: the grid that rounds down

A vector add in CUDA with `blocks = n / threads`, and the bounds check
inside the kernel. Run it for several sizes, and count the wrong elements.

This is Ticket 9. Predict the count for each size before you run. The first
run compiles the kernel, which takes 20 to 40 seconds.

**Your prediction** (write it before you run the cell)

- Crash?
- When?
- What?
- Which guard?

**What happened, and where you were wrong:**

In [ ]:
import cudalib

KERNEL = r"""
#include <ATen/cuda/CUDAContext.h>
#include <torch/extension.h>

__global__ void add(const float* a, const float* b, float* out, int n) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) out[i] = a[i] + b[i];
}

void launch(torch::Tensor a, torch::Tensor b, torch::Tensor out) {
  int n = a.numel();
  int threads = 256;
  int blocks = n / threads;                          // THE FAULT: rounds down
  if (blocks > 0)
    add<<<blocks, threads, 0, at::cuda::getCurrentCUDAStream()>>>(
        a.data_ptr<float>(), b.data_ptr<float>(), out.data_ptr<float>(), n);
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) { m.def("launch", &launch); }
"""
extension = cudalib.build_source('inc0_vector_add', KERNEL)

for n in (1024, 4096, 65536, 1000, 5000, 1009, 100):
  a, b = torch.rand(n, device=device), torch.rand(n, device=device)
  out = torch.zeros(n, device=device)
  extension.launch(a, b, out)
  wrong = (out != a + b).nonzero().flatten()
  where = f'{wrong[0].item()} to {wrong[-1].item()}' if len(wrong) else '-'
  print(f'n = {n:6d}: {len(wrong):4d} wrong   ({where})')

# Exercise 10: three mystery attentions

The module `mystery.py` holds three attention functions: `attention_a`,
`attention_b` and `attention_c`. Each one has one fault. **Do not open the
file.** Put each one into the model with `lab.use_attention`, and find the
fault from its behaviour.

For each function:

1. Run it on the default prompt, and compare with the reference.
2. Design a second experiment that makes the fault visible. Which variable do
   you change? The prompt? Its length? The number of queries?
3. Write your diagnosis: the fault, and the experiment that proved it.
4. Only then, open `mystery.py` and check.

A hint about the method: one function is correct for short prompts. One
function is wrong only when there is more than one query. One function breaks
everything, but you must still explain **how**.

In [ ]:
from mystery import attention_a, attention_b, attention_c

for name, attention in [('a', attention_a), ('b', attention_b), ('c', attention_c)]:
  lab.use_attention(model, attention)
  report(f'attention_{name}', lab.greedy(model, tokenizer, PROMPT, TOKENS), expected)
lab.use_attention(model, None)

**Your diagnosis**

- `attention_a`: the fault, and the experiment that proves it:
- `attention_b`: the fault, and the experiment that proves it:
- `attention_c`: the fault, and the experiment that proves it:

# Your fingerprint table

Fill in this table from what you saw, not from what you predicted.

| Fault | Crash? | When it shows | What it looks like | The guard |
|---|---|---|---|---|
| a tokenizer of another model | | | | |
| exp() in float16 | | | | |
| no causal mask | | | | |
| a memory-bound kernel on the FLOP ruler | | | | |
| no dtype at load | | | | |
| generate() with the model defaults | | | | |
| generate() with no length | | | | |
| no chat template | | | | |
| a grid that rounds down | | | | |